# 04 US split (M1b-4)
Runs on Colab (Drive mounted). Builds the US training set, a validation set and the US test set from the three Roboflow exports. Plan: PLAN.md (M1b-4) and the decisions of 2026-11-13.

In simple words: the 105 samrat_v5 valid+test images are the US test and are never removed. Any pool image that looks like a test image (dHash or ORB) is removed from the pool, together with its name copies. 10% of the remaining photo groups become validation (one image per group); validation images are checked the same way against the remaining training images. **No image is reviewed by hand; the leak check is automatic, so a copy that ORB and dHash both miss could remain.**

**Order:** run the cells top to bottom. The two long ORB cells only run when `RUN_LONG = True` (parameters cell). First run everything up to the timing cell, read the numbers, then set `RUN_LONG = True`. Both long cells save progress to Drive and continue after a disconnect.

**Image outputs:** the evidence-sheet cell and the box-check cell are tagged `remove_output` (images are CC BY, the repo is public). Run `python scripts/tag_notebook_cells.py notebooks/04_us_split.ipynb` before committing; Restart & Run All first.

In [1]:
import csv, hashlib, json, math, pickle, random, shutil, tarfile, time, zipfile
from collections import Counter
from pathlib import Path
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml  # preinstalled on Colab (PyYAML)
from PIL import Image
from google.colab import drive
from alpr import ccpd, roboflow as rf 

In [2]:
# Parameters
SEED = 0
VAL_FRAC = 0.10          # share of the pool's photo groups that become validation
DHASH_MAX = 6            # Hamming distance <= this = match (PLAN.md rule)
ORB_FEATURES = 500       # must equal the audit's value: descriptors are loaded from the audit cache
ORB_RATIO, ORB_MIN_GOOD = 0.75, 8
ORB_KEEP = 8             # pairs with at least this many inliers are saved to the hits files
T_MIN, T_MAX = 25, 42    # rule: T = highest random score + 1, clamped to this range (used only when T_OVERRIDE is None)
T_OVERRIDE = 42          # ORB threshold for cross-dataset pairs. The random-pair rule gave 25, too low: the highest chance score seen in the full search is 31 and unrelated pairs reach 39 inside samrat_v5; 42 is the audit's own samrat_v5 threshold. None = use the rule
N_CALIB = 5000           # random test-vs-florida/usa pairs for the calibration
AUDIT_T = {'samrat_v5': 42, 'florida_v6': 25, 'usa_v1': 40}   # audit thresholds, used for pairs inside one dataset
COPY_SETS = {'samrat_v5', 'florida_v6'}                       # sets where the name before ".rf." identifies the photo
NAME_COPY_CAP = 20       # name-copy expansion only for groups of at most this many images: the largest natural group is 10; the one 525-image florida_v6 group is a shared name stem, not one photo
RUN_LONG = True         # True runs the two long ORB cells; set it only after reading the timing cell
DRIVE = '/content/drive/MyDrive/alpr'
ARCHIVES = f'{DRIVE}/archives'
AUDIT = Path(DRIVE, 'reports/roboflow_audit')    # caches of notebook 03 (read only)
OUT = Path(DRIVE, 'reports/us_split')            # progress files of this notebook (private Drive folder, never the repo)
LOCAL = '/content/rf'
ZIPS = {
    'samrat_v5': 'License Plates.v5-augmented-license-plates.yolov8.zip',
    'florida_v6': 'License Plate Recognition.v6i.yolov8.zip',
    'usa_v1': 'License-plate-usa.v1i.yolov8.zip',
}
FOLDERS = ('train', 'valid', 'test')
TEST_FOLDERS = ('valid', 'test')   # of samrat_v5: the 105 US test images

In [3]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
%%bash
set -e
cd /content
if [ -d vehicle-plate-recognition ]; then git -C vehicle-plate-recognition pull --ff-only; else git clone https://github.com/medrees-1000/vehicle-plate-recognition.git; fi
pip install -q uv  # approved raw pip: only to bootstrap uv on Colab
uv pip install --system -q --no-deps -e /content/vehicle-plate-recognition

Updating d7fbcce..d40cf5a
Fast-forward
 scripts/tag_notebook_cells.py | 3 ++-
 1 file changed, 2 insertions(+), 1 deletion(-)


From https://github.com/medrees-1000/vehicle-plate-recognition
   d7fbcce..d40cf5a  master     -> origin/master


In [5]:
# setup: package, zips (copy + MD5 + extract), keys, labels, helpers # not in the top cell: the package exists only after the install cell (ImportError on the first run? restart the kernel once)
OUT.mkdir(parents=True, exist_ok=True)
IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
ROOT = {}
for name, fn in ZIPS.items():
    src, dst = Path(ARCHIVES, fn), Path(LOCAL, fn)
    assert src.exists(), f'{name}: {fn} not found in {ARCHIVES}'
    dst.parent.mkdir(parents=True, exist_ok=True)
    if not dst.exists():
        shutil.copyfile(src, dst)
    md5 = ccpd.file_md5(dst)
    assert md5 == ccpd.file_md5(src), f'{name}: VM copy differs from the Drive file'
    ROOT[name] = Path(LOCAL, name)
    if not ROOT[name].exists():
        with zipfile.ZipFile(dst) as z:
            z.extractall(ROOT[name])
    print(f'{name}: MD5 {md5}')

PLATE_IDS, N_CLS = {}, {}
for name in ZIPS:
    names = yaml.safe_load((ROOT[name] / 'data.yaml').read_text())['names']
    cls = dict(enumerate(names)) if isinstance(names, list) else {int(k): v for k, v in names.items()}
    PLATE_IDS[name] = frozenset(i for i, n in cls.items() if 'plate' in str(n).lower()); N_CLS[name] = len(cls)
    assert PLATE_IDS[name], f'{name}: no plate class in {cls}'
    print(name, 'classes', cls, '-> plate ids', set(PLATE_IDS[name]))

KEYS = {}   # "set/folder/name" -> image path
for name in ZIPS:
    for folder in FOLDERS:
        d = ROOT[name] / folder / 'images'
        if d.is_dir():
            for p in sorted(d.iterdir()):
                if p.suffix.lower() in IMG_EXT:
                    KEYS[rf.make_key(name, folder, p.name)] = p
TEST = [k for k in KEYS if rf.split_key(k)[0] == 'samrat_v5' and rf.split_key(k)[1] in TEST_FOLDERS]
_t = set(TEST)
POOL = [k for k in KEYS if k not in _t]
assert len(TEST) == 105, f'expected 105 test images, found {len(TEST)}'
FLUS = [k for k in POOL if rf.split_key(k)[0] != 'samrat_v5']   # florida_v6 + usa_v1, all folders
print(f'test {len(TEST)} | pool {len(POOL)} (expected 3,963) | florida_v6 + usa_v1 {len(FLUS)}')

def label_path(k):
    s, folder, n = rf.split_key(k)
    return ROOT[s] / folder / 'labels' / (Path(n).stem + '.txt')

def set_of(k): return rf.split_key(k)[0]

_last = {}
def progress(label, i, n, t0, every=30.0):
    now = time.time()
    if now - _last.get(label, 0.0) >= every or i == n:
        _last[label] = now
        print(f'  {label}: {i}/{n}, {now - t0:.0f} s elapsed', flush=True)

def dhash(gray):  # same 64-bit hash as notebook 03
    px = np.asarray(gray.resize((9, 8), Image.LANCZOS), dtype=np.int16)
    return int(''.join('1' if b else '0' for b in (px[:, 1:] > px[:, :-1]).ravel()), 2)

def orb_describe(path):
    g = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    kp, des = cv2.ORB_create(nfeatures=ORB_FEATURES).detectAndCompute(g, None)
    return (np.float32([k.pt for k in kp]), des) if des is not None and len(kp) else (np.zeros((0, 2), np.float32), None)

BF = cv2.BFMatcher(cv2.NORM_HAMMING)
def orb_score(a, b):
    """Inliers of the best rotation+scale+shift between two images' ORB keypoints (0 if too few matches). Same as notebook 03."""
    (pa, da), (pb, db) = a, b
    if da is None or db is None or len(da) < 2 or len(db) < 2:
        return 0
    good = [m[0] for m in BF.knnMatch(da, db, k=2) if len(m) == 2 and m[0].distance < ORB_RATIO * m[1].distance]
    if len(good) < ORB_MIN_GOOD:
        return 0
    M, inl = cv2.estimateAffinePartial2D(pa[[m.queryIdx for m in good]], pb[[m.trainIdx for m in good]], method=cv2.RANSAC, ransacReprojThreshold=4.0)
    return int(inl.sum()) if M is not None else 0

# ORB descriptors: the audit's cache (500 features), computed here only for a set that has none
DESC = {}
for name in ZIPS:
    pk = AUDIT / f'desc_{name}.pkl'
    if pk.exists():
        with open(pk, 'rb') as f:
            old = pickle.load(f)   # our own cache file from notebook 03, keys (folder, name)
        DESC.update({rf.make_key(name, s, n): v for (s, n), v in old.items()})
        print(f'{name}: descriptors loaded from the audit cache')
    else:
        t0 = time.time(); ks = [k for k in KEYS if set_of(k) == name]
        for i, k in enumerate(ks, 1):
            DESC[k] = orb_describe(KEYS[k]); progress(f'descriptors {name}', i, len(ks), t0)
missing = [k for k in KEYS if k not in DESC]
assert not missing, f'{len(missing)} images without descriptors, e.g. {missing[0]} (audit cache out of date? delete desc_<set>.pkl)'
def orb_keys(a, b): return orb_score(DESC[a], DESC[b])

samrat_v5: MD5 13e58f5a41f8e2e79375e9944c66dd4d
florida_v6: MD5 5970cdf35e4dbe188f2c82de92536ecb
usa_v1: MD5 47ab70118bd47344bacba2c1efe4101a
samrat_v5 classes {0: 'license-plate', 1: 'vehicle'} -> plate ids {0}
florida_v6 classes {0: 'License-Plate'} -> plate ids {0}
usa_v1 classes {0: 'License Plate'} -> plate ids {0}
test 105 | pool 3963 (expected 3,963) | florida_v6 + usa_v1 2984
samrat_v5: descriptors loaded from the audit cache
florida_v6: descriptors loaded from the audit cache
usa_v1: descriptors loaded from the audit cache


In [6]:
# time 500 real pairs
r = random.Random(SEED)
def timed(qs, ts, n=250):
    pairs = [(r.choice(qs), r.choice(ts)) for _ in range(n)]
    t0 = time.time(); [orb_keys(a, b) for a, b in pairs]
    return (time.time() - t0) / n
fl = [k for k in FLUS if set_of(k) == 'florida_v6']; us = [k for k in FLUS if set_of(k) == 'usa_v1']
per_fl, per_us, per_pool = timed(TEST, fl), timed(TEST, us), timed(POOL, POOL)
print(f'measured: test x florida_v6 {per_fl * 1000:.2f} ms/pair | test x usa_v1 {per_us * 1000:.2f} ms/pair | pool x pool {per_pool * 1000:.2f} ms/pair')
print(f'R1 (test-vs-pool ORB): {len(TEST) * len(fl):,} + {len(TEST) * len(us):,} pairs -> {len(TEST) * (len(fl) * per_fl + len(us) * per_us) / 60:.1f} min')
n_groups = len({(set_of(k), rf.source_of(rf.split_key(k)[2])) for k in POOL})
n_val = round(VAL_FRAC * n_groups)
print(f'R2 (validation vs train), upper bound before R1 removals: ~{n_val} val groups x ~{len(POOL):,} images = {n_val * len(POOL):,} pairs -> {n_val * len(POOL) * per_pool / 60:.1f} min')
print('Read these numbers, then set RUN_LONG = True in the parameters cell and re-run from there.')

measured: test x florida_v6 4.08 ms/pair | test x usa_v1 4.05 ms/pair | pool x pool 4.06 ms/pair
R1 (test-vs-pool ORB): 116,550 + 196,770 pairs -> 21.2 min
R2 (validation vs train), upper bound before R1 removals: ~100 val groups x ~3,963 images = 396,300 pairs -> 26.8 min
Read these numbers, then set RUN_LONG = True in the parameters cell and re-run from there.


In [7]:
# name groups per set
rows = []
for name in ZIPS:
    for folder in FOLDERS:
        ks = [k for k in POOL if set_of(k) == name and rf.split_key(k)[1] == folder]
        if ks:
            rows.append(dict(set=name, folder=folder, images=len(ks), name_groups=len({rf.source_of(rf.split_key(k)[2]) for k in ks})))
print(pd.DataFrame(rows).to_string(index=False))
AUD_ALL = pd.read_csv(AUDIT / 'orb_pairs.csv', on_bad_lines='skip').drop_duplicates(subset=['set', 'a', 'b', 'name_a', 'name_b'])   # every pair with >= 8 inliers
AUD = AUD_ALL[AUD_ALL['inliers'] >= AUD_ALL['set'].map(AUDIT_T)]
LINKS_AUDIT = [(rf.make_key(r.set, r.a, r.name_a), rf.make_key(r.set, r.b, r.name_b)) for r in AUD.itertuples()]
print(f'audit ORB links at the audit thresholds (inside one dataset, between folders): {len(LINKS_AUDIT)}')
G0, G1 = rf.group_ids(POOL, []), rf.group_ids(POOL, LINKS_AUDIT)
print(pd.DataFrame({name: {'images': sum(set_of(k) == name for k in POOL),
                           'name_groups': len({G0[k] for k in POOL if set_of(k) == name}),
                           'after_audit_links': len({G1[k] for k in POOL if set_of(k) == name})} for name in ZIPS}).T.to_string())
print('pool total: images', len(POOL), '| name groups', len(set(G0.values())), '| after audit links', len(set(G1.values())), '(groups can join across sets)')

       set folder  images  name_groups
 samrat_v5  train     979          245
florida_v6  train     972           97
florida_v6  valid      92           41
florida_v6   test      46           25
    usa_v1  train    1806          578
    usa_v1  valid      34           34
    usa_v1   test      34           33
audit ORB links at the audit thresholds (inside one dataset, between folders): 782
            images  name_groups  after_audit_links
samrat_v5      979          245                245
florida_v6    1110          112                111
usa_v1        1874          639                568
pool total: images 3963 | name groups 996 | after audit links 924 (groups can join across sets)


In [8]:
# test-vs-pool dHash
DH_CSV = OUT / 'dhash.csv'
DH = {}
if DH_CSV.exists():
    DH = {r.key: r.h for r in pd.read_csv(DH_CSV, dtype=str, keep_default_na=False).itertuples()}
todo = [k for k in KEYS if k not in DH]
print(f'{len(DH)} hashes cached, {len(todo)} to do')
t0 = time.time(); new = not DH_CSV.exists()
with open(DH_CSV, 'a', newline='') as f:
    wr = csv.writer(f)
    if new: wr.writerow(['key', 'h'])
    for i, k in enumerate(todo, 1):
        try:
            with Image.open(KEYS[k]) as im:
                h = format(dhash(im.convert('L')), '016x')
        except (OSError, ValueError):
            h = 'bad'
        DH[k] = h; wr.writerow([k, h]); progress('dhash', i, len(todo), t0)
print('unreadable images:', sum(v == 'bad' for v in DH.values()))
H = {k: int(v, 16) for k, v in DH.items() if v != 'bad'}
DH_TEST = [(p, t, (H[p] ^ H[t]).bit_count()) for t in TEST if t in H for p in POOL if p in H and (H[p] ^ H[t]).bit_count() <= DHASH_MAX]
print(f'pool images within Hamming <= {DHASH_MAX} of a test image: {len({p for p, _, _ in DH_TEST})} ({len(DH_TEST)} pairs)', Counter(set_of(p) for p, _, _ in DH_TEST))

4068 hashes cached, 0 to do
unreadable images: 0
pool images within Hamming <= 6 of a test image: 0 (0 pairs) Counter()


In [9]:
# calibrate T (5,000 random test-vs-florida/usa pairs)
r = random.Random(SEED + 7)
pairs = [(r.choice(TEST), r.choice(FLUS)) for _ in range(N_CALIB)]
t0 = time.time(); sc = [orb_keys(a, b) for a, b in pairs]
mx = max(sc)
T = min(T_MAX, max(T_MIN, mx + 1))
T_NOTE = f'highest random score {mx} -> T = {T}' + (f' (clamped to {T_MAX}: highest score + 1 = {mx + 1} is above the cap)' if mx + 1 > T_MAX else (f' (raised to the floor {T_MIN})' if mx + 1 < T_MIN else ''))
print(f'{N_CALIB} random pairs in {time.time() - t0:.0f} s; top 10 scores: {sorted(sc)[::-1][:10]}')
if T_OVERRIDE is not None:
    T_NOTE += f'; OVERRIDDEN: T = {T_OVERRIDE} (T_OVERRIDE). The random-pair maximum {mx} underestimated the tail of chance scores: unrelated pairs scored up to 31 against the pool and up to 39 inside samrat_v5; 42 is the audit threshold for samrat_v5'
    T = T_OVERRIDE
print('LOG:', T_NOTE)
json.dump(dict(n=N_CALIB, max=mx, T=T, note=T_NOTE), open(OUT / 'calibration.json', 'w'))

5000 random pairs in 25 s; top 10 scores: [14, 13, 10, 10, 10, 10, 9, 9, 9, 9]
LOG: highest random score 14 -> T = 25 (raised to the floor 25); OVERRIDDEN: T = 42 (T_OVERRIDE). The random-pair maximum 14 underestimated the tail of chance scores: unrelated pairs scored up to 31 against the pool and up to 39 inside samrat_v5; 42 is the audit threshold for samrat_v5


In [10]:
# R1 search: test-vs-pool ORB (long)
HITS1, DONE1 = OUT / 'orb_test_pool.csv', OUT / 'orb_test_pool.done.csv'
if RUN_LONG:
    rf.search_pairs(TEST, FLUS, orb_keys, HITS1, DONE1, ORB_KEEP, 'test-vs-pool ORB')
    print('R1 search finished')
else:
    print(f'Skipped: RUN_LONG is False. Left to do: {len(rf.pending(TEST, FLUS, DONE1))} of {len(TEST)} test images.')

  test-vs-pool ORB: 105/105 queries, 0 s elapsed
R1 search finished


In [11]:
# R1 removals
left = rf.pending(TEST, FLUS, DONE1)
assert not left, f'{len(left)} test images not searched yet: set RUN_LONG = True and run the cell above'
# samrat_v5 train vs the 105: the audit searched every samrat_v5 train image against valid and test (needs the audit log to be complete)
dq = pd.read_csv(AUDIT / 'orb_done.csv', dtype=str, on_bad_lines='skip').drop_duplicates()
n_tr = sum(1 for k in POOL if set_of(k) == 'samrat_v5')
for b in TEST_FOLDERS:
    got = dq[(dq['set'] == 'samrat_v5') & (dq['a'] == 'train') & (dq['b'] == b)]['name_a'].nunique()
    assert got == n_tr, f'audit search samrat_v5 train x {b} incomplete: {got}/{n_tr}'
fl_set = set(FLUS)
flag = {}   # pool key -> (reason, matched test key, score); strongest evidence first
def note(k, reason, t, s):
    if k not in flag: flag[k] = (reason, t, s)
for (q, t), s in sorted(rf.read_hits(HITS1).items(), key=lambda x: -x[1]):
    if t in fl_set and s >= T: note(t, 'orb', q, s)
for r in AUD_ALL[(AUD_ALL['set'] == 'samrat_v5') & (AUD_ALL['a'] == 'train')].itertuples():   # audit hits for samrat_v5 train x valid/test
    if r.b in TEST_FOLDERS and r.inliers >= T: note(rf.make_key('samrat_v5', 'train', r.name_a), 'orb_audit', rf.make_key('samrat_v5', r.b, r.name_b), int(r.inliers))
for p, t, d in DH_TEST: note(p, 'dhash', t, d)
SZ = Counter((set_of(k), rf.source_of(rf.split_key(k)[2])) for k in POOL)   # name-group sizes of the whole pool
def gkey(k): return (set_of(k), rf.source_of(rf.split_key(k)[2]))
def gsz(k): return SZ[gkey(k)]
def report_oversize(flagged, stage):   # flagged: key -> (reason, matched, score). These groups get no name-copy expansion
    seen = {}
    for k, (why, m, sc) in flagged.items():
        if gsz(k) > NAME_COPY_CAP: seen.setdefault(gkey(k), []).append(sc)
    for g, v in seen.items():
        print(f'{stage}: OVERSIZE group {g[0]}/{g[1]} has {SZ[g]} images (cap {NAME_COPY_CAP}): only the {len(v)} directly flagged image(s) removed, score(s) {sorted(v, reverse=True)[:5]}; no name-copy expansion')
copies = rf.expand_copies(set(flag), POOL, COPY_SETS, NAME_COPY_CAP, SZ)
report_oversize(flag, 'R1')
_n = Counter((set_of(k), rf.source_of(rf.split_key(k)[2])) for k in copies)
_seen = set()
for k, (why, t, sc) in flag.items():   # warning only; nothing is changed
    sk = (set_of(k), rf.source_of(rf.split_key(k)[2]))
    if _n.get(sk, 0) > 50 and sk not in _seen:
        _seen.add(sk); print(f'WARNING: one flagged image ({why}, score {sc}) pulls in {_n[sk]} name-copies from {sk[0]} (group of {_n[sk] + 1}) - check that the match is real')
REMOVED_R1 = [dict(key=k, set=set_of(k), reason=v[0], matched=v[1], score=v[2]) for k, v in flag.items()] + [dict(key=k, set=set_of(k), reason='name_copy', matched='', score='') for k in sorted(copies)]
R1 = {r['key'] for r in REMOVED_R1}
REMAIN = [k for k in POOL if k not in R1]
if 'GID' in globals() and set(GID) != set(REMAIN):
    print('WARNING: the remaining pool differs from the one the validation draw used: re-run the draw cell (and the R2 search)')
elif 'GID' in globals():
    print('the remaining pool is unchanged, so the validation draw and the R2 hits still apply')
print(f'T = {T}  ({T_NOTE})')
print(pd.DataFrame({name: {'removed_images': sum(r['set'] == name for r in REMOVED_R1),
                           'by_match': sum(r['set'] == name and r['reason'] != 'name_copy' for r in REMOVED_R1),
                           'name_copies': sum(r['set'] == name and r['reason'] == 'name_copy' for r in REMOVED_R1),
                           'groups': len({G1[r['key']] for r in REMOVED_R1 if r['set'] == name}),
                           'share_of_set': round(sum(r['set'] == name for r in REMOVED_R1) / sum(set_of(k) == name for k in POOL), 3)} for name in ZIPS}).T.to_string())
print(f'R1 total: {len(R1)} of {len(POOL)} pool images removed ({len(R1) / len(POOL):.1%}); {len({G1[k] for k in R1})} groups; remaining {len(REMAIN)}')
pd.DataFrame(REMOVED_R1).to_csv(f'{ARCHIVES}/us_yolo.removed.csv', index=False)   # rewritten with R2 added in the R2 cell

T = 42  (highest random score 14 -> T = 25 (raised to the floor 25); OVERRIDDEN: T = 42 (T_OVERRIDE). The random-pair maximum 14 underestimated the tail of chance scores: unrelated pairs scored up to 31 against the pool and up to 39 inside samrat_v5; 42 is the audit threshold for samrat_v5)
            removed_images  by_match  name_copies  groups  share_of_set
samrat_v5              0.0       0.0          0.0     0.0           0.0
florida_v6             0.0       0.0          0.0     0.0           0.0
usa_v1                 0.0       0.0          0.0     0.0           0.0
R1 total: 0 of 3963 pool images removed (0.0%); 0 groups; remaining 3963


In [12]:
# draw val groups (10% of the groups, seed 0)
GID = rf.group_ids(REMAIN, LINKS_AUDIT)
REPS, VAL_COPIES = rf.pick_val(GID, VAL_FRAC, SEED)
VAL = sorted(REPS.values()); _vg = set(REPS)
TRAIN_CAND = [k for k in REMAIN if GID[k] not in _vg]
print(pd.DataFrame({name: {'groups': len({GID[k] for k in REMAIN if set_of(k) == name}),
                           'val_images': sum(set_of(k) == name for k in VAL),
                           'val_copies_dropped': sum(set_of(k) == name for k in VAL_COPIES),
                           'train_candidates': sum(set_of(k) == name for k in TRAIN_CAND)} for name in ZIPS}).T.to_string())
print(f'groups in the remaining pool: {len(set(GID.values()))} | val groups drawn: {len(REPS)} | val images {len(VAL)} | train candidates {len(TRAIN_CAND)}')
# R2 dHash: each val image against the train candidates
DH_VAL = [(p, v, (H[p] ^ H[v]).bit_count()) for v in VAL if v in H for p in TRAIN_CAND if p in H and (H[p] ^ H[v]).bit_count() <= DHASH_MAX]
print(f'train candidates within Hamming <= {DHASH_MAX} of a val image: {len({p for p, _, _ in DH_VAL})}')

            groups  val_images  val_copies_dropped  train_candidates
samrat_v5      245          23                  69               887
florida_v6     111          12                  49              1049
usa_v1         568          57                 140              1677
groups in the remaining pool: 924 | val groups drawn: 92 | val images 92 | train candidates 3613
train candidates within Hamming <= 6 of a val image: 2


In [13]:
# R2 search: validation vs train candidates, ORB (long)
HITS2, DONE2 = OUT / 'orb_val_train.csv', OUT / 'orb_val_train.done.csv'
if RUN_LONG:
    _todo = rf.pending(VAL, TRAIN_CAND, DONE2)   # the done-log is tied to this exact target list; entries from another list are ignored
    print(f'{len(_todo)} of {len(VAL)} val images still to search for the current target list ({len(TRAIN_CAND)} train candidates)')
    rf.search_pairs(VAL, TRAIN_CAND, orb_keys, HITS2, DONE2, ORB_KEEP, 'val-vs-train ORB')
    print('R2 search finished')
else:
    print(f'Skipped: RUN_LONG is False. Left to do: {len(rf.pending(VAL, TRAIN_CAND, DONE2))} of {len(VAL)} val images.')

0 of 92 val images still to search for the current target list (3613 train candidates)
  val-vs-train ORB: 92/92 queries, 0 s elapsed
R2 search finished


In [14]:
# R2 removals
left = rf.pending(VAL, TRAIN_CAND, DONE2)
assert not left, f'{len(left)} val images not searched yet: set RUN_LONG = True and run the cell above'
cand, vset = set(TRAIN_CAND), set(VAL)
_gs = Counter(GID.values())
_big_g, _big_n = max(_gs[g] for g in REPS), max(gsz(v) for v in REPS.values())
print(f'val groups: {len(REPS)}; largest group (name + audit links, remaining pool) {_big_g}; largest name group {_big_n}; cap {NAME_COPY_CAP}: ' + ('OK, no validation group is larger than the cap' if max(_big_g, _big_n) <= NAME_COPY_CAP else 'WARNING: a validation group is larger than the cap'))
def thr(a, b): return AUDIT_T[set_of(a)] if set_of(a) == set_of(b) else T   # inside one dataset: the audit's threshold; across datasets: T
flag2, LINKS_VAL = {}, []
for (v, p), s in sorted(rf.read_hits(HITS2).items(), key=lambda x: -x[1]):
    if v in vset and p in cand and s >= thr(v, p):   # the hits file may also hold pairs of an earlier draw: only current val x candidates count
        flag2.setdefault(p, ('orb', v, s)); LINKS_VAL.append((v, p))
for p, v, d in DH_VAL: flag2.setdefault(p, ('dhash', v, d)); LINKS_VAL.append((v, p))
copies2 = rf.expand_copies(set(flag2), TRAIN_CAND, COPY_SETS, NAME_COPY_CAP, SZ)
report_oversize(flag2, 'R2')
REMOVED_R2 = [dict(key=k, set=set_of(k), reason='val_' + v[0], matched=v[1], score=v[2]) for k, v in flag2.items()] + [dict(key=k, set=set_of(k), reason='val_name_copy', matched='', score='') for k in sorted(copies2)]
R2 = {r['key'] for r in REMOVED_R2}
TRAIN = [k for k in TRAIN_CAND if k not in R2]
print(pd.DataFrame({name: {'removed_images': sum(r['set'] == name for r in REMOVED_R2),
                           'groups': len({GID[r['key']] for r in REMOVED_R2 if r['set'] == name}),
                           'share_of_pool': round(sum(r['set'] == name for r in REMOVED_R2) / sum(set_of(k) == name for k in POOL), 3)} for name in ZIPS}).T.to_string())
print(f'R2 removed {len(R2)} train images; total removed R1 + R2 = {len(R1) + len(R2)} of {len(POOL)} ({(len(R1) + len(R2)) / len(POOL):.1%}); val copies dropped {len(VAL_COPIES)}')
print(f'final: train {len(TRAIN)} | val {len(VAL)} | test {len(TEST)}')
REMOVED = REMOVED_R1 + REMOVED_R2
pd.DataFrame(REMOVED).to_csv(f'{ARCHIVES}/us_yolo.removed.csv', index=False)

val groups: 92; largest group (name + audit links, remaining pool) 10; largest name group 9; cap 20: OK, no validation group is larger than the cap
R2: OVERSIZE group florida_v6/i_jpg has 525 images (cap 20): only the 1 directly flagged image(s) removed, score(s) [49]; no name-copy expansion
            removed_images  groups  share_of_pool
samrat_v5             12.0     3.0          0.012
florida_v6            20.0     4.0          0.018
usa_v1               121.0    94.0          0.065
R2 removed 153 train images; total removed R1 + R2 = 153 of 3963 (3.9%); val copies dropped 258
final: train 3460 | val 92 | test 105


In [15]:
# name-group sizes and what-if
BIG = NAME_COPY_CAP
SZ = Counter((set_of(k), rf.source_of(rf.split_key(k)[2])) for k in POOL)   # name groups of the whole pool, before any removal
def gkey(k): return (set_of(k), rf.source_of(rf.split_key(k)[2]))
def gsz(k): return SZ[gkey(k)]
for name in ZIPS:
    s = sorted((v for (sn, _), v in SZ.items() if sn == name), reverse=True)
    print(f'{name}: {len(s)} name groups; 10 largest sizes {s[:10]}; median {np.median(s):.0f}; groups with more than {BIG} images: {sum(v > BIG for v in s)}')
for name in ZIPS:
    n_ok = sum(1 for (sn, _), v in SZ.items() if sn == name and v <= BIG); im_ok = sum(v for (sn, _), v in SZ.items() if sn == name and v <= BIG)
    n_big = sum(1 for (sn, _), v in SZ.items() if sn == name and v > BIG); im_big = sum(v for (sn, _), v in SZ.items() if sn == name and v > BIG)
    print(f'{name}: normal groups (size <= {BIG}): {n_ok} holding {im_ok} images | images in oversize groups: {im_big} ({n_big} group(s)); no photo count is claimed for oversize groups')
def used(r):
    if r['reason'].endswith('dhash'): return f'dHash <= {DHASH_MAX}'
    if r['reason'].startswith('val_'): return f'ORB >= {thr(r["matched"], r["key"])} ({"audit, same set" if set_of(r["matched"]) == set_of(r["key"]) else "T, cross-set"})'
    return f'ORB >= {T} (T)'
FLAGGED_GROUPS, REM_N = {}, Counter()   # (stage, set, source) -> strongest direct flag; removed images per such group
for stage, rows_ in (('R1', REMOVED_R1), ('R2', REMOVED_R2)):
    for r in rows_:
        g = (stage, *gkey(r['key'])); REM_N[g] += 1
        if not r['reason'].endswith('name_copy'): FLAGGED_GROUPS.setdefault(g, r)
tab = pd.DataFrame([dict(stage=g[0], set=g[1], group_size=gsz(r['key']), removed_from_group=REM_N[g], matched_set=set_of(r['matched']), reason=r['reason'], score=r['score'], threshold=used(r)) for g, r in FLAGGED_GROUPS.items()])
print(f'\ngroups with at least one directly flagged image (R1: matched = test image, R2: matched = val image): {len(tab)}')
print(tab.sort_values('group_size', ascending=False).to_string(index=False))
cp = [r for r in REMOVED_R1 + REMOVED_R2 if r['reason'].endswith('name_copy') and gsz(r['key']) > BIG]
print(f'\nfinal: train {len(TRAIN)} | val {len(VAL)} | test {len(TEST)}')
print(f'WHAT-IF, NOT APPLIED: without name expansion for groups > {BIG}, {len(cp)} name-copies stay in the pool -> train ~ {len(TRAIN) + len(cp)}', dict(Counter(r['set'] for r in cp)))
print('approximation: the val draw and the R2 search are not redone, so those images are not checked against val.')

samrat_v5: 245 name groups; 10 largest sizes [4, 4, 4, 4, 4, 4, 4, 4, 4, 4]; median 4; groups with more than 20 images: 0
florida_v6: 112 name groups; 10 largest sizes [525, 10, 9, 9, 9, 9, 9, 9, 9, 9]; median 5; groups with more than 20 images: 1
usa_v1: 639 name groups; 10 largest sizes [10, 9, 9, 9, 9, 9, 9, 7, 7, 7]; median 3; groups with more than 20 images: 0
samrat_v5: normal groups (size <= 20): 245 holding 979 images | images in oversize groups: 0 (0 group(s)); no photo count is claimed for oversize groups
florida_v6: normal groups (size <= 20): 111 holding 585 images | images in oversize groups: 525 (1 group(s)); no photo count is claimed for oversize groups
usa_v1: normal groups (size <= 20): 639 holding 1874 images | images in oversize groups: 0 (0 group(s)); no photo count is claimed for oversize groups

groups with at least one directly flagged image (R1: matched = test image, R2: matched = val image): 104
stage        set  group_size  removed_from_group matched_set  reas

In [ ]:
# contact sheet: removed name groups larger than 20
big = sorted(((gsz(r['key']), g, r) for g, r in FLAGGED_GROUPS.items() if gsz(r['key']) > BIG), key=lambda x: -x[0])
if not big:
    print(f'no removed group has more than {BIG} images')
else:
    rng = random.Random(SEED)
    fig, axes = plt.subplots(len(big), 8, figsize=(18, 2.4 * len(big)), squeeze=False)
    for ax in axes.ravel(): ax.axis('off')
    for row, (n, (stage, sn, src_), r) in zip(axes, big):
        members = [k for k in POOL if gkey(k) == (sn, src_) and k != r['key']]
        shown = [(r['matched'], 'val image' if stage == 'R2' else 'test image'), (r['key'], 'flagged')] + [(k, 'member') for k in rng.sample(members, min(6, len(members)))]
        for ax, (k, t) in zip(row, shown):
            ax.imshow(Image.open(KEYS[k]).convert('RGB')); ax.set_title(t, fontsize=8)
        row[0].set_title(f'{stage} {sn}: group of {n}, score {r["score"]} ({r["reason"]})\n{shown[0][1]}', fontsize=8, loc='left')
    fig.savefig(OUT / 'big_groups.png', dpi=100, bbox_inches='tight'); plt.show(); plt.close(fig)
    print(f'{len(big)} rows saved to {OUT / "big_groups.png"}')

In [ ]:
# evidence sheet: 20 highest test-vs-pool scores
top = sorted(((s, q, t) for (q, t), s in rf.read_hits(HITS1).items() if t in fl_set), reverse=True)[:20]
if not top:
    print('no test-vs-pool ORB hits to show')
else:
    fig, axes = plt.subplots(len(top), 2, figsize=(5, 2.2 * len(top)), squeeze=False)
    for ax in axes.ravel(): ax.axis('off')
    for row, (s, q, t) in zip(axes, top):
        row[0].imshow(Image.open(KEYS[q]).convert('RGB')); row[0].set_title(f'test, ORB {s}', fontsize=7)
        row[1].imshow(Image.open(KEYS[t]).convert('RGB')); row[1].set_title(f'{set_of(t)} {"(removed)" if s >= T else "(kept)"}', fontsize=7)
    fig.savefig(OUT / 'evidence_top20.png', dpi=100, bbox_inches='tight'); plt.show(); plt.close(fig)

In [18]:
# convert + build tar
ASSIGN = {**{k: 'train' for k in TRAIN}, **{k: 'val' for k in VAL}, **{k: 'test' for k in TEST}}
rf.check_split(ASSIGN, GID, LINKS_AUDIT + LINKS_VAL)   # raises if a group is on two sides or a link joins two sides
items = [rf.Item(k, KEYS[k], label_path(k), ASSIGN[k], PLATE_IDS[set_of(k)], N_CLS[set_of(k)]) for k in TEST + VAL + TRAIN]
t0 = time.time()
TAR = Path(LOCAL, 'us_yolo.tar')
kept, dropped = rf.build_tar(items, TAR)
print(f'tar built in {time.time() - t0:.0f} s')
print('kept per (set, side):', dict(sorted(kept.items())))
print('dropped per reason:', dict(Counter(r for *_, r in dropped)))
dr = {rf.make_key(s, f, n): r for s, f, n, r in dropped}
pd.DataFrame(dropped, columns=['set', 'folder', 'name', 'reason']).to_csv(f'{ARCHIVES}/us_yolo.dropped.csv', index=False)
why = {**{r['key']: 'removed:' + r['reason'] for r in REMOVED}, **{k: 'val_copy_dropped' for k in VAL_COPIES}}
G = pd.DataFrame([dict(set=set_of(k), folder=rf.split_key(k)[1], name=rf.split_key(k)[2], group_id=GID.get(k, ''),
                       split=(ASSIGN[k] if k in ASSIGN and k not in dr else ('dropped:' + dr[k] if k in dr else why.get(k, '?')))) for k in KEYS])
assert not (G['split'] == '?').any(), 'an image has no assignment'
G.to_csv(f'{ARCHIVES}/us_yolo.groups.csv', index=False)
Gf = G[G['split'].isin(['train', 'val', 'test'])].assign(source=lambda d: d['name'].map(rf.source_of))
Gf['over'] = [SZ[(s, so)] > NAME_COPY_CAP for s, so in zip(Gf['set'], Gf['source'])]
a_ = Gf[~Gf['over']].groupby(['set', 'split']).agg(images_normal_groups=('name', 'size'), source_photos_normal_groups=('source', 'nunique'))
b_ = Gf[Gf['over']].groupby(['set', 'split']).size().rename('images_in_oversize_groups')
print(a_.join(b_, how='outer').fillna(0).astype(int).to_string())   # no photo count for oversize groups
print('all images by final status:'); print(G.groupby(['set', 'split']).size().to_string())
shutil.copyfile(TAR, f'{ARCHIVES}/us_yolo.tar')

tar built in 4 s
kept per (set, side): {('florida_v6', 'train'): 967, ('florida_v6', 'val'): 12, ('samrat_v5', 'test'): 105, ('samrat_v5', 'train'): 875, ('samrat_v5', 'val'): 23, ('usa_v1', 'train'): 1531, ('usa_v1', 'val'): 57}
dropped per reason: {'no_plate': 87}
                  images_normal_groups  source_photos_normal_groups  images_in_oversize_groups
set        split                                                                              
florida_v6 train                   447                           86                        520
           val                      12                           12                          0
samrat_v5  test                    105                          105                          0
           train                   875                          219                          0
           val                      23                           23                          0
usa_v1     train                  1531                          565 

'/content/drive/MyDrive/alpr/archives/us_yolo.tar'

In [19]:
# copy to /content + MD5
CHECK = Path('/content/us_yolo_check.tar')
shutil.copyfile(f'{ARCHIVES}/us_yolo.tar', CHECK)
m_local, m_back = ccpd.file_md5(TAR), ccpd.file_md5(CHECK)
assert m_local == m_back, f'MD5 differs after the copy: {m_local} vs {m_back}'
print('us_yolo.tar MD5', m_back, '(record it in PLAN.md)')
cnt = Counter()
with tarfile.open(CHECK) as t:
    for n in t.getnames():
        if n.startswith(('images/', 'labels/')): cnt[tuple(n.split('/')[:2])] += 1
exp = Counter(); [exp.update({('images', s): n, ('labels', s): n}) for (_, s), n in kept.items()]
assert cnt == exp, f'tar contents differ from the kept counts: {cnt} vs {exp}'
print('files per kind/side:', dict(sorted(cnt.items())))

us_yolo.tar MD5 1ce491ae6a7accefdb0b9156a30f2acc (record it in PLAN.md)
files per kind/side: {('images', 'test'): 105, ('images', 'train'): 3373, ('images', 'val'): 92, ('labels', 'test'): 105, ('labels', 'train'): 3373, ('labels', 'val'): 92}


In [ ]:
# visual box check
r = random.Random(SEED)
fig, axes = plt.subplots(3, 6, figsize=(18, 9))
areas, area_set = [], []
with tarfile.open(CHECK) as t:
    names = t.getnames()
    for row, side in zip(axes, ('train', 'val', 'test')):
        imgs = sorted(n for n in names if n.startswith(f'images/{side}/'))
        for ax, n in zip(row, r.sample(imgs, min(6, len(imgs)))):
            im = Image.open(t.extractfile(n)).convert('RGB'); ax.imshow(im); ax.axis('off'); ax.set_title(f'{side} {Path(n).name[:22]}', fontsize=7)
            for line in t.extractfile(n.replace('images/', 'labels/', 1).rsplit('.', 1)[0] + '.txt').read().decode().splitlines():
                _, cx, cy, w, h = map(float, line.split())
                ax.add_patch(plt.Rectangle(((cx - w / 2) * im.width, (cy - h / 2) * im.height), w * im.width, h * im.height, fill=False, ec='red', lw=1.5))
        for ax in row[len(imgs):]: ax.axis('off')
    for n in names:   # box-size summary over all labels
        if n.startswith('labels/'):
            for line in t.extractfile(n).read().decode().splitlines(): areas.append(float(line.split()[3]) * float(line.split()[4])); area_set.append(Path(n).name.split('__')[0])
plt.show(); plt.close(fig)
print(f'{len(areas)} boxes; box area share of the image: min {min(areas):.4f}, median {np.median(areas):.4f}, max {max(areas):.4f}')
for name in ZIPS:
    a_ = np.array([a for a, sn in zip(areas, area_set) if sn == name])
    print(f'{name}: {len(a_)} boxes; area share < 0.0005: {int((a_ < 0.0005).sum())}; > 0.5: {int((a_ > 0.5).sum())}')


## Before committing
1. Restart & Run All (with `RUN_LONG = True`), then save.
2. `python scripts/tag_notebook_cells.py notebooks/04_us_split.ipynb` (tags the evidence-sheet cell and the box-check cell, `remove_output`).
3. Check that no image output is left in the saved notebook.

## Facts for PLAN.md (from the diagnostic cells)
- florida_v6 has one name group of 525 images; every other group has at most 10 (samrat_v5 max 4, florida_v6 next largest 10, usa_v1 max 10). The 525 members are different photos (dashboard views, parking lots): a shared name stem, not one photo. `NAME_COPY_CAP = 20`: larger groups get no name-copy expansion.
- The ORB match that pulled the 525 in was a chance match: score 49, cross-set (florida_v6 train image vs a usa_v1 validation image), T = 42; the scenes differ. So a cross-set score above T can still be a false match.
- Box sizes in the final tar: samrat_v5 113 of 1,326 boxes have an area share under 0.0005 (tiny plates); florida_v6 has 8 and usa_v1 2 boxes over 0.5.
